# 01 — Exploratory Data Analysis
**Purpose:** Understand the raw Census data before any feature engineering or modeling.
We ask three questions:
1. What does the distribution of each raw variable look like?
2. Are there meaningful differences across the five Atlanta metro counties?
3. How much missing data exists, and does it cluster in any particular county or year?

All observations here directly inform downstream modeling decisions documented in `02_feature_engineering.ipynb`.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore")
import plotly.io as pio
pio.renderers.default = "iframe"


conn = sqlite3.connect("../housing_pulse.db")
df = pd.read_sql("SELECT * FROM census_tracts", conn)
conn.close()

print(f"Shape: {df.shape}")
print(f"Years: {sorted(df['data_year'].unique())}")
print(f"Counties: {sorted(df['county_name'].unique())}")
df.head()

Shape: (2012, 20)
Years: [np.int64(2022), np.int64(2024)]
Counties: ['Clayton', 'Cobb', 'DeKalb', 'Fulton', 'Gwinnett']


,total_renter_hh,severely_burdened_hh,median_income,median_rent,vacant_units,total_units,white_pop,total_pop,NAME,state,county,tract,county_name,rent_burden_pct,vacancy_rate,rent_to_income_ratio,white_share,geoid,data_year,pulled_at
0,913,487,154808.0,1611.0,62,2520,4349,5160,Census Tract 1; Fulton County; Georgia,13,121,000100,Fulton,0.5334,0.0246,0.1249,0.8428,13121000100,2022,2026-03-08T18:53:04.554510
1,415,29,120982.0,1674.0,111,1331,1755,2233,Census Tract 2.01; Fulton County; Georgia,13,121,000201,Fulton,0.0699,0.0834,0.1660,0.7859,13121000201,2022,2026-03-08T18:53:04.554510
2,261,14,182409.0,3071.0,0,1681,3294,4106,Census Tract 2.02; Fulton County; Georgia,13,121,000202,Fulton,0.0536,0.0000,0.2020,0.8022,13121000202,2022,2026-03-08T18:53:04.554510
3,352,12,127500.0,2089.0,230,1294,1574,2014,Census Tract 4; Fulton County; Georgia,13,121,000400,Fulton,0.0341,0.1777,0.1966,0.7815,13121000400,2022,2026-03-08T18:53:04.554510
4,1229,140,103485.0,1997.0,557,2973,1943,4385,Census Tract 5.01; Fulton County; Georgia,13,121,000501,Fulton,0.1139,0.1874,0.2316,0.4431,13121000501,2022,2026-03-08T18:53:04.554510


## 1. Dataset Overview
We have two ACS 5-year estimate vintages:
- **2022** (covering 2018–2022) — baseline year
- **2024** (covering 2020–2024) — latest available as of January 2026

Each vintage covers the same 1,006 census tracts across 5 Atlanta metro counties,
giving us 2,012 rows total. Working at the **tract level** (not county) is deliberate —
county-level aggregates mask extreme within-county heterogeneity, as we demonstrate below.

In [ ]:
# Tract counts by year and county
print(df.groupby(["data_year","county_name"]).size().unstack(fill_value=0))

county_name  Clayton  Cobb  DeKalb  Fulton  Gwinnett
data_year                                           
2022              70   186     203     327       220
2024              70   186     203     327       220


## 2. Missing Data Audit
Census tracts with very small renter populations often have suppressed estimates
(reported as -666666666 by the API, converted to NaN in our pipeline).
We need to quantify how much data is missing before deciding whether to impute or drop.

In [ ]:
# Missing data by feature and year
missing = df.groupby("data_year")[
    ["rent_burden_pct","vacancy_rate","rent_to_income_ratio",
     "median_income","median_rent","white_share"]
].apply(lambda x: x.isna().mean().mul(100).round(2))
print("Missing % by feature and year:")
print(missing)

Missing % by feature and year:
           rent_burden_pct  vacancy_rate  rent_to_income_ratio  median_income  \
data_year                                                                       
2022                  1.59           0.5                 11.83           1.19   
2024                  0.99           0.5                  9.64           1.49   

           median_rent  white_share  
data_year                            
2022             11.13          0.3  
2024              8.95          0.3  


In [ ]:
# Missing data clusters by county?
missing_by_county = df.groupby("county_name")[
    ["rent_burden_pct","median_income","median_rent"]
].apply(lambda x: x.isna().mean().mul(100).round(1))
print(missing_by_county)

             rent_burden_pct  median_income  median_rent
county_name                                             
Clayton                  1.4            1.4          2.9
Cobb                     1.3            0.0         12.1
DeKalb                   1.5            1.5          5.7
Fulton                   1.8            2.9         14.2
Gwinnett                 0.2            0.0          8.4


## 3. Distribution of Raw Features
We examine each key variable for skew, outliers, and plausible ranges.
This determines whether normalization or clipping is needed in feature engineering.

In [ ]:
numeric_cols = ["median_income","median_rent","rent_burden_pct",
                "vacancy_rate","rent_to_income_ratio"]

for col in numeric_cols:
    subset = df[col].dropna()
    print(f"{col:30s}  mean={subset.mean():,.1f}  median={subset.median():,.1f}  "
          f"std={subset.std():,.1f}  p5={subset.quantile(0.05):,.1f}  "
          f"p95={subset.quantile(0.95):,.1f}")

median_income                   mean=96,670.4  median=86,732.0  std=47,028.1  p5=39,837.2  p95=191,175.4
median_rent                     mean=1,491.7  median=1,441.0  std=450.0  p5=886.9  p95=2,230.8
rent_burden_pct                 mean=0.2  median=0.2  std=0.2  p5=0.0  p95=0.6
vacancy_rate                    mean=0.1  median=0.1  std=0.1  p5=0.0  p95=0.2
rent_to_income_ratio            mean=0.2  median=0.2  std=0.1  p5=0.1  p95=0.3


In [ ]:
# Histogram grid for all key features
fig = px.histogram(
    df[df["data_year"]==2024].dropna(subset=["median_rent"]),
    x="median_rent", nbins=50,
    title="Distribution of Median Rent (2024 ACS)",
    labels={"median_rent": "Median Monthly Rent ($)"},
    color_discrete_sequence=["#3498db"]
)
fig.update_layout(plot_bgcolor="white", paper_bgcolor="white")
fig.show()

In [ ]:
fig = px.histogram(
    df[df["data_year"]==2024].dropna(subset=["median_income"]),
    x="median_income", nbins=50,
    title="Distribution of Median Household Income (2024 ACS)",
    labels={"median_income": "Median Household Income ($)"},
    color_discrete_sequence=["#2ecc71"]
)
fig.update_layout(plot_bgcolor="white", paper_bgcolor="white")
fig.show()

## 4. County-Level Comparison
**Why this matters:** If county-level averages look similar, working at the tract level
adds complexity without insight. If counties differ materially, tract-level analysis
is essential to avoid ecological fallacy.

In [ ]:
county_summary = df[df["data_year"]==2024].groupby("county_name").agg(
    tracts=("geoid","count"),
    avg_rent_burden=("rent_burden_pct","mean"),
    avg_median_income=("median_income","mean"),
    avg_median_rent=("median_rent","mean"),
    avg_vacancy=("vacancy_rate","mean")
).round(3)
print(county_summary)

             tracts  avg_rent_burden  avg_median_income  avg_median_rent  \
county_name                                                                
Clayton          70            0.320          61925.304         1237.235   
Cobb            186            0.217         114561.016         1678.482   
DeKalb          203            0.293          92394.226         1512.592   
Fulton          327            0.251         109229.038         1597.728   
Gwinnett        220            0.277          94429.041         1731.293   

             avg_vacancy  
county_name               
Clayton            0.086  
Cobb               0.054  
DeKalb             0.129  
Fulton             0.092  
Gwinnett           0.042  


In [ ]:
fig = px.bar(
    county_summary.reset_index().sort_values("avg_rent_burden", ascending=False),
    x="county_name", y="avg_rent_burden",
    title="Average Rent Burden by County (2024 ACS)",
    labels={"avg_rent_burden": "Avg Rent Burden Rate", "county_name": "County"},
    color="avg_rent_burden",
    color_continuous_scale="Reds"
)
fig.update_layout(plot_bgcolor="white", paper_bgcolor="white")
fig.show()

**Observation:** Clayton County's median rent burden is materially higher than the
other four counties — consistent with its lower-income profile (~$59,800 median household
income vs. $90,000+ in Cobb and Fulton).

Fulton's wide **within-county spread** reflects the Buckhead-to-southwest-Atlanta income
stratification. Both neighborhoods sit within the same county but are structurally
different markets — Fulton's income inequality ratio exceeds 24x (FRED: 2020RATIO013121).

**This validates the decision to work at the tract level rather than county level.**
County medians for Fulton are essentially meaningless for housing policy — they average
two completely different markets into a single number.

## 5. Year-over-Year Change (2022 → 2024)
With two ACS vintages, we can observe directional trends at the tract level.
This is the same signal the drift monitor (PSI) quantifies formally.

In [ ]:
df22 = df[df["data_year"]==2022].set_index("geoid")[["median_rent","median_income","rent_burden_pct"]]
df24 = df[df["data_year"]==2024].set_index("geoid")[["median_rent","median_income","rent_burden_pct"]]
delta = (df24 - df22).dropna()
delta.columns = ["rent_change","income_change","burden_change"]

print("Median change 2022 → 2024:")
print(f"  Median rent:    ${delta['rent_change'].median():+.0f}/month")
print(f"  Median income:  ${delta['income_change'].median():+.0f}/year")
print(f"  Rent burden:    {delta['burden_change'].median():+.3f} (proportion)")

fig = px.histogram(delta["rent_change"].clip(-300,300), nbins=60,
    title="Tract-Level Rent Change 2022 → 2024",
    labels={"value":"Monthly Rent Change ($)"},
    color_discrete_sequence=["#e74c3c"])
fig.add_vline(x=0, line_dash="dash", line_color="black")
fig.update_layout(plot_bgcolor="white", paper_bgcolor="white")
fig.show()

Median change 2022 → 2024:
  Median rent:    $+186/month
  Median income:  $+6420/year
  Rent burden:    +0.022 (proportion)


## 6. Key EDA Takeaways

1. **Missing data is sparse and non-systematic** — concentrated in small tracts with
   suppressed Census estimates. Safe to drop rather than impute.

2. **Rent and income are right-skewed** — normalization using percentile clipping
   (rather than z-score) is more robust to outliers at the high end.

3. **Clayton County is the highest-burden county** by average rent burden rate.
   This will surface as a concentration of High/Critical DRI tiers in `features.py`.

4. **Fulton County has extreme within-county spread** — validating tract-level analysis.

5. **Rents rose faster than incomes 2022 → 2024** — the rent distribution shift
   produces the PSI=0.298 RETRAIN flag observed in `monitor.py`. This is real signal,
   not noise.